# Lab 4 — Performance e funcionamento interno do Spark

**Objetivos**
- Entender job, stage e task, e o que é um shuffle
- Ler um plano de execução com `explain()`
- Escolher a estratégia de join correta (broadcast vs sort-merge)
- Detectar e corrigir *data skew*
- Saber o que o AQE faz automaticamente
- Decidir quando usar cache e quando evitar UDFs

---
## 1. Setup

In [1]:
# !pip install -q pyspark==4.0.1 pyarrow

In [2]:
import io, re, time, contextlib
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Aula 4 - Performance")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.adaptive.enabled", "false")   # desligado de propósito; ligamos na seção 8
    .config("spark.ui.showConsoleProgress", "false") # esconde as barras de progresso
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark   :", spark.version)
print("Núcleos :", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/28 20:17:19 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/10/28 20:17:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 20:17:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark   : 4.0.1
Núcleos : 24
Spark UI: http://10.23.20.50:4040


Funções auxiliares usadas no notebook:

- `timer` mede o tempo e coloca um nome no job (aparece na Spark UI).
- `run` força a execução **sem trazer dados para o driver**, usando o formato `noop`. É o jeito correto de medir performance: `show()` lê só algumas linhas e `collect()` mede também a transferência para o Python.
- `count_shuffles` conta quantos `Exchange` (shuffles) existem no plano.

In [3]:
@contextlib.contextmanager
def timer(label):
    spark.sparkContext.setJobDescription(label)
    t0 = time.perf_counter()
    yield # quando finaliza
    print(f"{label:<45} {time.perf_counter() - t0:6.2f} s")
    spark.sparkContext.setJobDescription(None)

def run(df):
    df.write.format("noop").mode("overwrite").save()

def plan_text(df, mode="simple"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode=mode)
    return buf.getvalue()

def count_shuffles(df):
    return len(re.findall(r"\bExchange\b", plan_text(df)))

def partition_balance(df, *keys, n=16, label=""):
    """Redistribui por chave e mostra o tamanho da maior partição vs a mediana."""
    s = (df.repartition(n, *keys)
           .groupBy(F.spark_partition_id()).count()
           .agg(F.max("count").alias("max"), F.percentile_approx("count", 0.5).alias("median"))
           .first())
    print(f"{label:<12} maior = {s['max']:>9,} | mediana = {s['median']:>9,} | razão = {s['max'] / s['median']:.1f}x")

---
## 2. Dataset grande

Vamos gerar três tabelas no estilo da aula 3, mas em escala:

| Tabela | Linhas | Papel |
|--------|--------|-------|
| `orders` | 5 milhões | Fatos (pedidos) |
| `customers` | 200 mil | Dimensão média |
| `products` | 1 mil | Dimensão pequena |

`spark.range()` gera os dados já distribuídos, sem passar pelo Python. Salvamos em Parquet para que as leituras seguintes sejam I/O real.

> Se a sua máquina for modesta (ou no Colab gratuito), reduza `N_ORDERS` para 2 milhões.

In [4]:
N_ORDERS    = 5_000_000
N_CUSTOMERS = 200_000
N_PRODUCTS  = 1_000
REGIONS     = ["North", "Northeast", "Midwest", "Southeast", "South"]
CATEGORIES  = ["Computers", "Peripherals", "Audio", "Storage", "Cables"]
BASE        = "/tmp/aula4"

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

orders_gen = (
    spark.range(N_ORDERS).withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.rand(1) * N_CUSTOMERS).cast("int"))
    .withColumn("product_id",  (F.rand(2) * N_PRODUCTS).cast("int"))
    .withColumn("quantity",    (F.rand(3) * 5 + 1).cast("int"))
    .withColumn("price",       F.round(F.rand(4) * 490 + 10, 2))
    .withColumn("region",      pick(REGIONS, 5))
    .withColumn("order_date",  F.date_add(F.to_date(F.lit("2024-01-01")), (F.rand(6) * 365).cast("int")))
)

customers_gen = (
    spark.range(N_CUSTOMERS).withColumnRenamed("id", "customer_id")
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("segment", pick(["Consumer", "Corporate", "Small Business"], 11))
    .withColumn("signup_year", (F.rand(12) * 6 + 2019).cast("int"))
)

products_gen = (
    spark.range(N_PRODUCTS).withColumnRenamed("id", "product_id")
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.concat(F.lit("Product "), F.col("product_id")))
    .withColumn("category", pick(CATEGORIES, 21))
)

with timer("Gerar e salvar as 3 tabelas"):
    orders_gen.write.mode("overwrite").parquet(f"{BASE}/orders")
    customers_gen.write.mode("overwrite").parquet(f"{BASE}/customers")
    products_gen.write.mode("overwrite").parquet(f"{BASE}/products")

orders    = spark.read.parquet(f"{BASE}/orders")
customers = spark.read.parquet(f"{BASE}/customers")
products  = spark.read.parquet(f"{BASE}/products")

orders.show(5)

Gerar e salvar as 3 tabelas                     2.96 s
+--------+-----------+----------+--------+------+---------+----------+
|order_id|customer_id|product_id|quantity| price|   region|order_date|
+--------+-----------+----------+--------+------+---------+----------+
| 2291666|      73092|       417|       1| 98.89|  Midwest|2024-12-19|
| 2291667|     186251|       989|       3|317.22|    North|2024-01-28|
| 2291668|     140692|       483|       4|183.57|    North|2024-01-20|
| 2291669|      32718|       607|       5| 113.4|    North|2024-07-17|
| 2291670|       1265|       401|       5|411.26|Northeast|2024-04-30|
+--------+-----------+----------+--------+------+---------+----------+
only showing top 5 rows


### 2.1 Pandas vs Spark com 5 milhões de linhas

Mesma consulta nas duas ferramentas: receita total por região.
O resultado costuma surpreender.

In [5]:
with timer("Pandas: ler Parquet + groupby"):
    pdf = pd.read_parquet(f"{BASE}/orders")
    pdf["revenue"] = pdf["price"] * pdf["quantity"]
    res_pd = pdf.groupby("region")["revenue"].sum()

with timer("Spark: ler Parquet + groupBy"):
    res_sp = (
        spark.read.parquet(f"{BASE}/orders")
        .groupBy("region")
        .agg(F.sum(F.col("price") * F.col("quantity")).alias("revenue"))
        .collect()
    )

print(f"\nMemória usada pelo DataFrame Pandas: {pdf.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
del pdf

Pandas: ler Parquet + groupby                   0.49 s
Spark: ler Parquet + groupBy                    0.88 s

Memória usada pelo DataFrame Pandas: 660 MB


**Como interpretar:** em uma única máquina, com dados que cabem na RAM, o Pandas frequentemente empata ou ganha. O Spark paga um custo fixo (planejamento, JVM, coordenação de tasks).

O Spark compensa quando:
- os dados **não cabem na memória** de uma máquina (o Pandas simplesmente quebra);
- existe um **cluster**: o mesmo código escala para dezenas de máquinas sem mudanças;
- o pipeline lê de fontes distribuídas (data lake, S3, HDFS).

Por isso, performance em Spark é menos "qual é mais rápido" e mais **"como evitar trabalho desnecessário"**. É isso que o resto da aula aborda.

---
## 3. Como o Spark executa: job, stage e task

Quando você chama uma **ação**, o Spark transforma o plano em trabalho físico:

| Conceito | O que é | Quantos |
|----------|---------|---------|
| **Job** | Trabalho disparado por uma ação (`count`, `write`, `collect`) | 1 por ação (às vezes mais) |
| **Stage** | Bloco de operações que rodam sem trocar dados entre partições | Novo stage a cada shuffle |
| **Task** | Execução de um stage sobre **uma** partição | 1 por partição |

```
  df.filter(...).withColumn(...).groupBy("region").sum()  .write  <- ação
                                                            │
                                                           JOB
            ┌───────────────────────────────────────────────┴────────────┐
         STAGE 1                                                       STAGE 2
   ler + filter + withColumn                                         soma final
   + soma parcial por partição                                       por região
   [T1] [T2] [T3] [T4]   ─────────── SHUFFLE (Exchange) ──────────▶  [T1] [T2] [T3]
```

### Transformações narrow vs wide

| Tipo | Cada partição de saída depende de... | Exemplos | Shuffle? |
|------|-------------------------------------|----------|----------|
| **Narrow** | uma partição de entrada | `select`, `filter`, `withColumn`, `union` | Não |
| **Wide** | várias partições de entrada | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` | Sim |

```
   NARROW                          WIDE (shuffle)
   P1 ──▶ P1'                      P1 ──┬──▶ P1'
   P2 ──▶ P2'                      P2 ──┼──▶ P2'
   P3 ──▶ P3'                      P3 ──┴──▶ P3'
```

O **shuffle** é a operação mais cara do Spark: os dados são serializados, gravados em disco, transferidos pela rede e lidos de novo. **Otimizar Spark é, em grande parte, reduzir shuffles e o volume que passa por eles.**

### 3.1 Lendo o plano com `explain()`

No plano físico, todo shuffle aparece como **`Exchange`**. O plano é lido **de baixo para cima**.

In [6]:
query = (
    orders
    .filter(F.col("price") > 100)                                 # narrow
    .withColumn("revenue", F.col("price") * F.col("quantity"))    # narrow
    .groupBy("region")                                            # wide
    .agg(F.sum("revenue").alias("revenue"))
)

query.explain(mode="formatted")
print("Shuffles no plano:", count_shuffles(query))

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [quantity#21, price#22, region#23]
Batched: true
Location: InMemoryFileIndex [file:/tmp/aula4/orders]
PushedFilters: [IsNotNull(price), GreaterThan(price,100.0)]
ReadSchema: struct<quantity:int,price:double,region:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]

(3) Filter [codegen id : 1]
Input [3]: [quantity#21, price#22, region#23]
Condition : (isnotnull(price#22) AND (price#22 > 100.0))

(4) Project [codegen id : 1]
Output [2]: [region#23, (price#22 * cast(quantity#21 as double)) AS revenue#74]
Input [3]: [quantity#21, price#22, region#23]

(5) HashAggregate [codegen id : 1]
Input [2]: [region#23, revenue#74]
Keys [1]: [region#23]
Functions [1]: [partial_sum(revenue#74)]
Aggregate Attributes [1]: [sum#85]
Result

Pontos para observar no plano:

- **`FileScan parquet`** com `PushedFilters`: o filtro `price > 100` foi empurrado para a leitura (*predicate pushdown*).
- **`ReadSchema`**: só as colunas usadas são lidas (*column pruning*). Vantagem do Parquet, que é colunar.
- **`HashAggregate (partial)`** antes do `Exchange`: cada partição soma localmente, e só os totais parciais atravessam o shuffle.
- **`Exchange hashpartitioning(region, 200)`**: o shuffle, com 200 partições de saída.

In [7]:
# Uma query com ordenação + agregação + join: quantos shuffles?
q2 = (
    orders
    .join(products, "product_id")
    .groupBy("category").agg(F.count("*").alias("n"))
    .orderBy(F.desc("n"))
)
print("Shuffles:", count_shuffles(q2))
q2.explain()

Shuffles: 2
== Physical Plan ==
*(4) Sort [n#89L DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(n#89L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=255]
   +- *(3) HashAggregate(keys=[category#30], functions=[count(1)])
      +- Exchange hashpartitioning(category#30, 200), ENSURE_REQUIREMENTS, [plan_id=251]
         +- *(2) HashAggregate(keys=[category#30], functions=[partial_count(1)])
            +- *(2) Project [category#30]
               +- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false
                  :- *(2) Filter isnotnull(product_id#20)
                  :  +- *(2) ColumnarToRow
                  :     +- FileScan parquet [product_id#20] Batched: true, DataFilters: [isnotnull(product_id#20)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<product_id:int>
                  +- BroadcastExchange HashedRelationBroadc

---
## 4. Partições

A partição é a unidade de paralelismo: **uma task processa uma partição**. Poucas partições deixam núcleos ociosos. Partições demais geram overhead de agendamento e muitos arquivos pequenos.

| Momento | Quem decide o número de partições | Configuração |
|---------|----------------------------------|--------------|
| Leitura de arquivos | Tamanho dos arquivos | `spark.sql.files.maxPartitionBytes` (128 MB) |
| Após um shuffle | Configuração fixa (sem AQE) | `spark.sql.shuffle.partitions` (200) |
| Manual | Você | `repartition(n)` / `coalesce(n)` |

Regra prática: partições entre **100 MB e 200 MB**, e pelo menos **2 a 4 vezes o número de núcleos**.

In [8]:
print("Partições na leitura:", orders.rdd.getNumPartitions())

# Quantas linhas há em cada partição?
(
    orders
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show()
)

Partições na leitura: 24
+---------+------+
|partition| count|
+---------+------+
|        0|208334|
|        1|208333|
|        2|208333|
|        3|208333|
|        4|208333|
|        5|208333|
|        6|208333|
|        7|208334|
|        8|208334|
|        9|208333|
|       10|208333|
|       11|208334|
|       12|208333|
|       13|208333|
|       14|208333|
|       15|208333|
|       16|208333|
|       17|208334|
|       18|208334|
|       19|208333|
+---------+------+
only showing top 20 rows


In [9]:
# Efeito de spark.sql.shuffle.partitions em uma agregação
agg = orders.groupBy("customer_id").agg(F.sum("price").alias("total"))

for n in [200, 2, spark.sparkContext.defaultParallelism * 2]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    with timer(f"groupBy com shuffle.partitions = {n}"):
        run(agg)

spark.conf.set("spark.sql.shuffle.partitions", 200)

groupBy com shuffle.partitions = 200            0.83 s
groupBy com shuffle.partitions = 2              0.55 s
groupBy com shuffle.partitions = 48             0.32 s


Com dados pequenos, 200 partições geram 200 tasks minúsculas, e o overhead domina. Com 2, poucas tasks fazem todo o trabalho e os núcleos ficam ociosos. O valor ideal depende do volume, e é exatamente isso que o AQE ajusta sozinho (seção 8).

### `repartition` vs `coalesce`

In [10]:
print("repartition(8):")
orders.repartition(8).explain()

print("coalesce(2):")
orders.coalesce(2).explain()

repartition(8):
== Physical Plan ==
Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=466]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...


coalesce(2):
== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#18L,customer_id#19,product_id#20,quantity#21,price#22,region#23,order_date#24] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/aula4/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:int,product_id:int,quantity:int,price:double,region:string,ord...




| | `repartition(n)` | `coalesce(n)` |
|-|------------------|---------------|
| Shuffle | Sim (`Exchange`) | Não |
| Pode aumentar partições | Sim | Não, só reduz |
| Distribuição | Uniforme | Pode ficar desbalanceada |
| Uso típico | Antes de joins pesados, por chave | Reduzir arquivos antes de escrever |

---
## 5. Estratégias de join

Um join precisa que linhas com a mesma chave estejam na mesma máquina. Há duas formas principais de conseguir isso:

```
  SORT-MERGE JOIN (padrão)                 BROADCAST HASH JOIN
  as duas tabelas passam por shuffle        a tabela pequena é copiada para todos

  orders ──shuffle──┐                       products (pequena)
                    ├── sort + merge          │ cópia │ cópia │ cópia
  customers ─shuffle┘                       [orders P1] [orders P2] [orders P3]
                                            nenhum shuffle da tabela grande
```

| Estratégia | Quando usar | Custo |
|------------|-------------|-------|
| **Broadcast Hash Join** | Um lado é pequeno (padrão: < 10 MB) | Sem shuffle do lado grande; o lado pequeno vai para a memória de cada executor |
| **Sort-Merge Join** | Duas tabelas grandes | Shuffle + ordenação dos dois lados |
| **Shuffle Hash Join** | Um lado médio, sem ordenação | Shuffle dos dois lados, hash do menor |

O Spark escolhe broadcast automaticamente abaixo de `spark.sql.autoBroadcastJoinThreshold`. Vamos desligar esse limite para comparar.

In [11]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # desliga o broadcast automático

smj = orders.join(products, "product_id")
print("Sort-merge — shuffles:", count_shuffles(smj))
with timer("Join orders x products (sort-merge)"):
    run(smj)

bhj = orders.join(F.broadcast(products), "product_id")
print("\nBroadcast  — shuffles:", count_shuffles(bhj))
with timer("Join orders x products (broadcast)"):
    run(bhj)

Sort-merge — shuffles: 2
Join orders x products (sort-merge)             1.47 s

Broadcast  — shuffles: 0
Join orders x products (broadcast)              0.44 s


In [12]:
# Confirme no plano qual estratégia foi usada
print([l.strip() for l in plan_text(smj).splitlines() if "Join" in l])
print([l.strip() for l in plan_text(bhj).splitlines() if "Join" in l])

['+- *(5) SortMergeJoin [product_id#20], [product_id#28], Inner']
['+- *(2) BroadcastHashJoin [product_id#20], [product_id#28], Inner, BuildRight, false']


Três formas equivalentes de pedir broadcast:

```python
orders.join(F.broadcast(products), "product_id")      # função
orders.join(products.hint("broadcast"), "product_id") # hint na API
spark.sql("SELECT /*+ BROADCAST(p) */ * FROM orders o JOIN products p USING (product_id)")
```

> Cuidado: fazer broadcast de uma tabela grande pode estourar a memória do driver e dos executores. Use para dimensões pequenas.

In [13]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10MB")   # restaura o padrão

In [ ]:
spark.stop()